
## What is a Pipeline?

A Pipeline in scikit-learn is a tool that chains multiple data processing steps together into a single object. Think of it as an assembly line: data flows through preprocessing steps and finally through a machine learning model, all in one streamlined process.

## Why Use Pipelines?

**Problems with Manual Preprocessing:**

1. **Code Repetition**: You must manually apply the same preprocessing steps to training, validation, and test data
2. **Error-Prone**: Easy to forget a step or apply transformations in the wrong order
3. **Data Leakage Risk**: Accidentally fitting transformers on test data leads to overly optimistic performance estimates
4. **Messy Code**: Lots of intermediate variables and scattered preprocessing logic
5. **Difficult to Deploy**: Must remember and recreate all preprocessing steps in production

**Benefits of Pipelines:**

1. **Cleaner Code**: All preprocessing and modeling in one object
2. **Prevents Data Leakage**: Automatically fits transformers only on training data
3. **Easier Cross-Validation**: Works seamlessly with GridSearchCV and cross_val_score
4. **Reproducible**: The same transformation steps are guaranteed to be applied consistently
5. **Deployment-Ready**: Single pipeline object contains everything needed for predictions

## Learning Objectives

In this notebook, you will:

1. See the manual preprocessing approach and its problems
2. Learn how to build a pipeline step-by-step
3. Compare both approaches side-by-side
4. Use pipelines with cross-validation and hyperparameter tuning
5. Understand best practices for pipeline usage


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')


## Step 1: Load and Explore Data

We'll use a synthetic dataset simulating customer purchase behavior based on age, salary, and city.


In [2]:
# Create a more realistic sample dataset
np.random.seed(42)
n_samples = 200

# Generate synthetic data
age = np.random.randint(18, 65, n_samples)
salary = np.random.randint(30000, 150000, n_samples)
cities = np.random.choice(['New York', 'Chicago', 'San Francisco', 'Boston', 'Seattle'], n_samples)

# Create target variable based on features (with some noise)
# Higher age and salary increase purchase probability
purchase_prob = (age / 65) * 0.3 + (salary / 150000) * 0.5 + np.random.random(n_samples) * 0.2
purchased = (purchase_prob > 0.5).astype(int)

# Create DataFrame
data = pd.DataFrame({
    'age': age,
    'salary': salary,
    'city': cities,
    'purchased': purchased
})

print("Dataset shape:", data.shape)
print("\nFirst few rows:")
print(data.head(10))
print("\nData types:")
print(data.dtypes)
print("\nTarget distribution:")
print(data['purchased'].value_counts())


Dataset shape: (200, 4)

First few rows:
   age  salary           city  purchased
0   56   82733       New York          1
1   46   95318         Boston          1
2   32  139953        Seattle          1
3   60  119474        Seattle          1
4   25   53664       New York          0
5   38   97172  San Francisco          1
6   56  115616        Chicago          1
7   36  123264       New York          1
8   40  145386        Chicago          1
9   28   56736        Chicago          0

Data types:
age           int32
salary        int32
city         object
purchased     int32
dtype: object

Target distribution:
purchased
1    150
0     50
Name: count, dtype: int64


### Split Data into Training and Test Sets


In [3]:
# Separate features and target
X = data.drop(columns=['purchased'])
y = data['purchased']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nFeature columns:")
print(X_train.columns.tolist())


Training set size: (160, 3)
Test set size: (40, 3)

Feature columns:
['age', 'salary', 'city']


## Step 2: Method 1 - Manual Preprocessing (Non-Pipeline Approach)

Let's first see how we traditionally handle preprocessing without pipelines. This will help us appreciate why pipelines are valuable.


### Manual Step 1: Identify Feature Types


In [4]:
# Identify numeric and categorical features
numeric_features = ['age', 'salary']
categorical_features = ['city']

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


Numeric features: ['age', 'salary']
Categorical features: ['city']


### Manual Step 2: Create and Fit Transformers on Training Data


In [5]:
# Create separate transformers for numeric and categorical features
scaler = StandardScaler()
encoder = OneHotEncoder(drop='first', sparse_output=False)

# IMPORTANT: Fit transformers ONLY on training data to avoid data leakage
scaler.fit(X_train[numeric_features])
encoder.fit(X_train[categorical_features])

print("Scaler fitted on training data")
print("Encoder fitted on training data")
print("Encoder categories:", encoder.categories_)


Scaler fitted on training data
Encoder fitted on training data
Encoder categories: [array(['Boston', 'Chicago', 'New York', 'San Francisco', 'Seattle'],
      dtype=object)]


### Manual Step 3: Transform Training Data


In [6]:
# Transform numeric features
X_train_numeric_scaled = scaler.transform(X_train[numeric_features])

# Transform categorical features
X_train_categorical_encoded = encoder.transform(X_train[categorical_features])

# Combine transformed features
X_train_processed = np.hstack([X_train_numeric_scaled, X_train_categorical_encoded])

print("Training data shape after preprocessing:", X_train_processed.shape)
print("Original features: 3 (age, salary, city)")
print("Processed features:", X_train_processed.shape[1], "(2 numeric + encoded categorical)")


Training data shape after preprocessing: (160, 6)
Original features: 3 (age, salary, city)
Processed features: 6 (2 numeric + encoded categorical)


### Manual Step 4: Train the Model


In [7]:
# Create and train logistic regression model
model_manual = LogisticRegression(max_iter=1000, random_state=42)
model_manual.fit(X_train_processed, y_train)

print("Model trained on manually preprocessed training data")


Model trained on manually preprocessed training data


### Manual Step 5: Transform Test Data and Predict

CRITICAL: We must apply the SAME transformations (fitted on training data) to test data.


In [8]:
# Transform test data using the SAME fitted transformers
X_test_numeric_scaled = scaler.transform(X_test[numeric_features])
X_test_categorical_encoded = encoder.transform(X_test[categorical_features])

# Combine transformed test features
X_test_processed = np.hstack([X_test_numeric_scaled, X_test_categorical_encoded])

# Make predictions
y_pred_manual = model_manual.predict(X_test_processed)

# Evaluate
accuracy_manual = accuracy_score(y_test, y_pred_manual)
print("Manual Method Test Accuracy:", accuracy_manual)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_manual))


Manual Method Test Accuracy: 0.8

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.50      0.56        10
           1       0.84      0.90      0.87        30

    accuracy                           0.80        40
   macro avg       0.73      0.70      0.71        40
weighted avg       0.79      0.80      0.79        40



### Problems with Manual Preprocessing

Notice what we had to do:

1. **Multiple steps**: Create transformers, fit them, transform training data, train model, transform test data separately
2. **Multiple objects to track**: scaler, encoder, model - all must be kept in sync
3. **Repetitive code**: Same transformation logic for training and test data
4. **Error-prone**: Easy to accidentally fit transformers on test data (data leakage)
5. **Hard to deploy**: Must save and load scaler, encoder, and model separately
6. **Difficult cross-validation**: Cannot easily use GridSearchCV or cross_val_score

Let's see how pipelines solve these problems.


## Step 3: Method 2 - Pipeline Approach

Now let's solve the same problem using a pipeline. Notice how much cleaner and safer this is.


### Pipeline Step 1: Define ColumnTransformer

ColumnTransformer allows us to apply different transformations to different columns.


In [9]:
# Define preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),  # Scale numeric features
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)  # Encode categorical
    ],
    remainder='drop'  # Drop any columns not specified
)

print("Preprocessor defined:")
print("- Numeric features will be scaled using StandardScaler")
print("- Categorical features will be one-hot encoded")


Preprocessor defined:
- Numeric features will be scaled using StandardScaler
- Categorical features will be one-hot encoded


### Pipeline Step 2: Build the Complete Pipeline

Chain preprocessing and model into a single pipeline object.


In [10]:
# Create pipeline: preprocessing -> model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # First: preprocessing
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))  # Second: model
])

print("Pipeline created with steps:")
for name, step in pipeline.steps:
    print(f"  - {name}: {step.__class__.__name__}")


Pipeline created with steps:
  - preprocessor: ColumnTransformer
  - classifier: LogisticRegression


### Pipeline Step 3: Fit the Pipeline

One call to `fit()` handles everything:

- Fits preprocessing transformers on training data
- Transforms training data
- Trains the model


In [11]:
# Fit the entire pipeline on training data
pipeline.fit(X_train, y_train)

print("Pipeline fitted successfully!")
print("Behind the scenes:")
print("  1. StandardScaler fitted on training age and salary")
print("  2. OneHotEncoder fitted on training city values")
print("  3. Training data transformed")
print("  4. LogisticRegression fitted on transformed training data")


Pipeline fitted successfully!
Behind the scenes:
  1. StandardScaler fitted on training age and salary
  2. OneHotEncoder fitted on training city values
  3. Training data transformed
  4. LogisticRegression fitted on transformed training data


### Pipeline Step 4: Make Predictions

One call to `predict()` handles everything:
- Transforms test data using fitted transformers
- Makes predictions using trained model


In [12]:
# Predict on test data (preprocessing automatically applied)
y_pred_pipeline = pipeline.predict(X_test)

# Evaluate
accuracy_pipeline = accuracy_score(y_test, y_pred_pipeline)
print("Pipeline Method Test Accuracy:", accuracy_pipeline)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_pipeline))


Pipeline Method Test Accuracy: 0.8

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.50      0.56        10
           1       0.84      0.90      0.87        30

    accuracy                           0.80        40
   macro avg       0.73      0.70      0.71        40
weighted avg       0.79      0.80      0.79        40



### Compare Both Methods


In [13]:
print("="*70)
print("COMPARISON: Manual vs Pipeline Approach")
print("="*70)
print(f"Manual Method Accuracy:   {accuracy_manual:.4f}")
print(f"Pipeline Method Accuracy: {accuracy_pipeline:.4f}")
print(f"\nResults match: {np.allclose(accuracy_manual, accuracy_pipeline)}")
print("\nCode Comparison:")
print("  Manual Method:  ~30 lines, 3 objects to manage (scaler, encoder, model)")
print("  Pipeline Method: ~10 lines, 1 object to manage (pipeline)")
print("\nBenefits of Pipeline:")
print("  + Cleaner code: All preprocessing and modeling in one object")
print("  + Safer: No risk of fitting on test data")
print("  + Easier deployment: Save/load one object instead of three")
print("  + Better for cross-validation: Works seamlessly with GridSearchCV")
print("  + More maintainable: One place to update preprocessing logic")


COMPARISON: Manual vs Pipeline Approach
Manual Method Accuracy:   0.8000
Pipeline Method Accuracy: 0.8000

Results match: True

Code Comparison:
  Manual Method:  ~30 lines, 3 objects to manage (scaler, encoder, model)
  Pipeline Method: ~10 lines, 1 object to manage (pipeline)

Benefits of Pipeline:
  + Cleaner code: All preprocessing and modeling in one object
  + Safer: No risk of fitting on test data
  + Easier deployment: Save/load one object instead of three
  + Better for cross-validation: Works seamlessly with GridSearchCV
  + More maintainable: One place to update preprocessing logic


## Step 4: Advanced - Cross-Validation with Pipelines

One of the biggest advantages of pipelines is seamless integration with cross-validation. Let's see how easy it is.


### Cross-Validation Score

With pipelines, `cross_val_score` handles all preprocessing automatically within each fold.


In [14]:
# Perform 5-fold cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')

print("5-Fold Cross-Validation Scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nMean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\nWhy this is safe with pipelines:")
print("  - Each fold fits transformers only on training portion")
print("  - Test portion never seen during preprocessing fit")
print("  - No data leakage between folds")


5-Fold Cross-Validation Scores:
  Fold 1: 0.9688
  Fold 2: 0.9688
  Fold 3: 0.9062
  Fold 4: 0.9375
  Fold 5: 0.9688

Mean CV Accuracy: 0.9500 (+/- 0.0250)

Why this is safe with pipelines:
  - Each fold fits transformers only on training portion
  - Test portion never seen during preprocessing fit
  - No data leakage between folds


## Step 5: Hyperparameter Tuning with GridSearchCV

Pipelines make hyperparameter tuning straightforward. We can tune both preprocessing and model parameters.


### Define Parameter Grid

Parameter names in pipeline follow the pattern: 'step_name__parameter_name'


In [15]:
# Define parameter grid for the classifier
# Use 'classifier__' prefix to access parameters of the classifier step
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
    'classifier__penalty': ['l1', 'l2'],  # Regularization type
    'classifier__solver': ['liblinear']  # Solver that supports both l1 and l2
}

print("Parameter grid defined:")
print("  - Testing 5 different C values (regularization strength)")
print("  - Testing 2 penalty types (l1 and l2)")
print(f"  - Total combinations to test: {len(param_grid['classifier__C']) * len(param_grid['classifier__penalty'])}")


Parameter grid defined:
  - Testing 5 different C values (regularization strength)
  - Testing 2 penalty types (l1 and l2)
  - Total combinations to test: 10


### Create and Fit GridSearchCV


In [16]:
# Create GridSearchCV object
grid_search = GridSearchCV(
    pipeline,  # The pipeline to optimize
    param_grid,  # Parameters to search
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',  # Optimization metric
    n_jobs=-1,  # Use all available cores
    verbose=1,  # Show progress
    return_train_score=True  # Track training scores too
)

print("GridSearchCV configured:")
print("  - Pipeline:    Includes preprocessing + LogisticRegression")
print("  - CV folds:    5")
print("  - Metric:      Accuracy")
print("  - Parameters:  classifier__C, classifier__penalty")
print("\nFitting GridSearchCV...")

# Fit grid search (this will take a moment)
grid_search.fit(X_train, y_train)

print("\nGrid search complete!")


GridSearchCV configured:
  - Pipeline:    Includes preprocessing + LogisticRegression
  - CV folds:    5
  - Metric:      Accuracy
  - Parameters:  classifier__C, classifier__penalty

Fitting GridSearchCV...
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Grid search complete!


### Examine Best Parameters and Scores


In [17]:
print("="*70)
print("GRID SEARCH RESULTS")
print("="*70)
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Show top 5 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

print("\nTop 5 Parameter Combinations:")
print(results_df[['param_classifier__C', 'param_classifier__penalty', 
                   'mean_test_score', 'std_test_score', 'rank_test_score']].head())


GRID SEARCH RESULTS

Best Parameters: {'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV Score: 0.9563

Top 5 Parameter Combinations:
   param_classifier__C param_classifier__penalty  mean_test_score  \
1                 0.01                        l2          0.95625   
4                 1.00                        l1          0.95000   
3                 0.10                        l2          0.94375   
5                 1.00                        l2          0.94375   
7                10.00                        l2          0.94375   

   std_test_score  rank_test_score  
1        0.031869                1  
4        0.025000                2  
3        0.036443                3  
5        0.036443                3  
7        0.036443                3  


### Evaluate Best Model on Test Set

The best model is automatically refitted on the entire training set.


In [18]:
# Get the best pipeline (already refitted on full training data)
best_pipeline = grid_search.best_estimator_

# Predict on test set
y_pred_best = best_pipeline.predict(X_test)

# Evaluate
test_accuracy_best = accuracy_score(y_test, y_pred_best)

print("="*70)
print("FINAL MODEL EVALUATION")
print("="*70)
print(f"Best CV Score (from grid search): {grid_search.best_score_:.4f}")
print(f"Test Set Accuracy:                {test_accuracy_best:.4f}")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))


FINAL MODEL EVALUATION
Best CV Score (from grid search): 0.9563
Test Set Accuracy:                0.8500

Best Parameters:
  classifier__C: 0.01
  classifier__penalty: l2
  classifier__solver: liblinear

Confusion Matrix:
[[ 6  4]
 [ 2 28]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.60      0.67        10
           1       0.88      0.93      0.90        30

    accuracy                           0.85        40
   macro avg       0.81      0.77      0.78        40
weighted avg       0.84      0.85      0.84        40



## Summary: Why Use Pipelines?

### Key Advantages Demonstrated

1. **Cleaner Code**
   - Manual approach: Multiple objects to manage (scaler, encoder, model)
   - Pipeline approach: Single pipeline object
   
2. **Prevents Data Leakage**
   - Transformers always fit only on training data
   - Automatic proper handling in cross-validation
   
3. **Easier Cross-Validation**
   - Works seamlessly with cross_val_score and GridSearchCV
   - No need to manually handle preprocessing in each fold
   
4. **Reproducible**
   - Same preprocessing steps guaranteed for all data
   - Single object to save/load for deployment
   
5. **Less Error-Prone**
   - No risk of forgetting preprocessing steps
   - No accidental fitting on test data
   - Consistent transformation order

### When to Use Pipelines

**Always use pipelines when:**

- You have preprocessing steps (scaling, encoding, imputation, etc.)
- You're doing cross-validation or hyperparameter tuning
- You need to deploy your model
- You want clean, maintainable code

**Pipeline is not needed when:**

- You have no preprocessing (rare)
- You're doing exploratory data analysis only


## Best Practices and Tips

### Pipeline Naming Convention

When accessing parameters in GridSearchCV, use the format: `step_name__parameter_name`

```python
# For a pipeline with steps named 'preprocessor' and 'classifier':
param_grid = {
    'classifier__C': [0.1, 1, 10],  # Access classifier's C parameter
    'preprocessor__num__with_mean': [True, False]  # Access nested parameter
}
```

### Accessing Pipeline Components

```python
# Get a specific step from the pipeline
preprocessor = pipeline.named_steps['preprocessor']
model = pipeline.named_steps['classifier']

# Get fitted transformers
scaler = preprocessor.named_transformers_['num']
encoder = preprocessor.named_transformers_['cat']
```

### Common ColumnTransformer Parameters

- `remainder='drop'`: Drop columns not specified (default)
- `remainder='passthrough'`: Keep unspecified columns as-is
- `n_jobs=-1`: Parallelize transformations (useful for large datasets)

### Saving and Loading Pipelines

```python
import joblib

# Save entire pipeline (preprocessing + model)
joblib.dump(pipeline, 'pipeline_model.pkl')

# Load and use
loaded_pipeline = joblib.load('pipeline_model.pkl')
predictions = loaded_pipeline.predict(new_data)
```

##  Independent Study

### Exercise 1: Add More Preprocessing
Try adding more preprocessing steps to the pipeline:

- Add SimpleImputer for handling missing values
- Add PolynomialFeatures for feature engineering
- Add SelectKBest for feature selection

### Exercise 2: Try Different Models
Replace LogisticRegression with:

- RandomForestClassifier
- GradientBoostingClassifier
- SVC (Support Vector Classifier)

Compare their performance using the same pipeline structure.

### Exercise 3: Tune Preprocessing Parameters
Extend the GridSearchCV to tune preprocessing parameters:

- Test different strategies for OneHotEncoder (drop='first' vs drop=None)
- Try StandardScaler vs MinMaxScaler vs RobustScaler
- Test different polynomial degrees if you add PolynomialFeatures

### Exercise 4: Use Real Data
Apply what you learned to a real dataset:

- Load a dataset from sklearn.datasets or Kaggle
- Build a complete pipeline with appropriate preprocessing
- Use GridSearchCV to find optimal hyperparameters
- Evaluate on a held-out test set

### Additional Resources

- Sklearn Pipeline Documentation: https://scikit-learn.org/stable/modules/compose.html
- Sklearn ColumnTransformer Guide: https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data
- GridSearchCV Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html
